In [180]:
# create a class
import torch
import torch.nn as nn   

class Model(nn.Module):              # inherit from nn.Module (gives automatic weight/gradient handling)

    def __init__(self, num_features):
        super().__init__()           # MUST call this - activates nn.Module's machinery (never skip)

        # nn.Linear(in, out) - a fully-connected layer that AUTO-creates weights & bias
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()  # squashes output to 0-1 (a probability)

    def forward(self, features):   
        out = self.linear(features)  # weighted sum (z = Wx + b), done automatically
        out = self.sigmoid(out)      # sigmoid activation 
        return out
        

In [181]:
# create a dataset
features=torch.rand(10,5)

In [182]:
features.shape[1]

5

In [183]:
# create a model 
model=Model(features.shape[1])

# this returns 10 predictions (one probability per sample)
model(features)

tensor([[0.3672],
        [0.4134],
        [0.4116],
        [0.4860],
        [0.4485],
        [0.3667],
        [0.3758],
        [0.4293],
        [0.4455],
        [0.4668]], grad_fn=<SigmoidBackward0>)

#### Why model(features) and not model.forward(features)?

#### Both work, but model(features) is the correct professional way. When you call model(features), PyTorch runs forward() plus important internal hooks (needed for some features). Always use model(features), not model.forward(features) directly.

In [184]:
# weights nn.Linear created automatically 
model.linear.weight

Parameter containing:
tensor([[-0.4114,  0.1188, -0.0545, -0.4230, -0.1495]], requires_grad=True)

In [185]:
# bias nn.Linear created automatically
model.linear.bias

Parameter containing:
tensor([0.1556], requires_grad=True)

In [186]:
! pip install torchinfo

In [187]:
from torchinfo import summary 
summary (model,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [188]:
#  2-layer model - adding a HIDDEN layer with ReLU
class Model1(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.linear1 = nn.Linear(num_features, 3)   # hidden layer: features -> 3 neurons
        self.relu = nn.ReLU()                        # ReLU activation (for hidden layers)
        self.linear2 = nn.Linear(3, 1)               # output layer: 3 -> 1
        self.sigmoid = nn.Sigmoid()                  # sigmoid -> probability (0 to 1)

    def forward(self, features):
        # data flows through each layer MANUALLY, one step at a time
        out = self.linear1(features)    # hidden layer
        out = self.relu(out)            # activation
        out = self.linear2(out)         # output layer
        out = self.sigmoid(out)         # probability
        return out
        # this manual chain gets LONG with more layers so will use nn.Sequential later 

In [189]:
features=torch.rand(10,5)

In [190]:
# create a model 
model1=Model1(features.shape[1])


model1(features)

tensor([[0.4543],
        [0.4541],
        [0.4543],
        [0.4543],
        [0.4541],
        [0.4485],
        [0.4543],
        [0.4522],
        [0.4539],
        [0.4512]], grad_fn=<SigmoidBackward0>)

In [191]:
model1.linear1.weight

Parameter containing:
tensor([[ 0.1116,  0.0245, -0.4467,  0.0568, -0.4461],
        [-0.1869, -0.1502,  0.3152, -0.3030,  0.0866],
        [-0.1375,  0.4429,  0.3042,  0.3372,  0.0152]], requires_grad=True)

In [192]:
model1.linear2.weight

Parameter containing:
tensor([[ 0.0100, -0.1425, -0.0356]], requires_grad=True)

In [193]:
model1.linear1.bias

Parameter containing:
tensor([-0.4120, -0.3501, -0.3530], requires_grad=True)

In [194]:
model1.linear2.bias

Parameter containing:
tensor([-0.1832], requires_grad=True)

In [195]:
summary (model1,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model1                                   [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### Model with a Hidden Layer

## Upgrading from a single layer to a **2-layer network**:
- `linear1` → hidden layer (num_features → 3 neurons)
- `relu` → activation for the hidden layer (avoids vanishing gradient)
- `linear2` → output layer (3 → 1)
- `sigmoid` → converts output to a probability (0–1)

**Data flow:** `features → linear1 → ReLU → linear2 → Sigmoid → prediction`

 **Note the problem:** as the network gets deeper, the `forward()` method becomes a 
long chain of manual steps (`out = self.linear1(out)`, `out = self.relu(out)`, ...). 
This gets tedious and error-prone with many layers.

 **Solution:** will use a **container** like `nn.Sequential` to stack layers automatically, 
so the forward pass becomes a single line instead of many.

In [196]:
import torch
import torch.nn as nn   

class Model2(nn.Module):          # inherit from nn.Module 

    def __init__(self, num_features):
        super().__init__()        # MUST call this - activates nn.Module's features
        
        # nn.Sequential - stack layers in order, data flows through them top to bottom
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),   # layer 1: num_features inputs → 3 neurons
            nn.ReLU(),                     # activation (ReLU)
            nn.Linear(3, 1),               # layer 2: 3 inputs → 1 output
            nn.Sigmoid()                   # sigmoid → probability (0 to 1)
        )

    def forward(self, features):
        out = self.network(features)   # run data through all the layers
        return out

#### nn.Module — the base class ALL PyTorch models inherit from. It automatically handles weight tracking, gradients, GPU moving, saving/loading. You get all this for free.
#### super().__init__() — don't forget it! This activates nn.Module's machinery. Without it, your model breaks (weights won't be tracked).
#### nn.Linear(in, out) — a fully-connected layer. It automatically creates and manages the weights and bias for you! 
#### nn.Linear(5, 3) = 5 inputs → 3 outputs, with a (5×3) weight matrix + 3 biases, all created automatically.

#### Remember doing torch.rand() manually in our training pipeline? nn.Linear does ALL that 

#### nn.ReLU() — the ReLU activation. Used in hidden layers to avoid vanishing gradient.
#### nn.Sigmoid() — squashes output to 0–1 (probability). Used in the output layer for binary classification.
#### nn.Sequential(...) — stacks layers so data flows through them in order: input → Linear → ReLU → Linear → Sigmoid → output.
#### forward() — defines how data flows through the model. we can  call it indirectly via model(features).

In [197]:
features=torch.rand(10,5)

In [198]:
# create a model 
model2=Model2(features.shape[1])


model2(features)

tensor([[0.3158],
        [0.3304],
        [0.3328],
        [0.3284],
        [0.3252],
        [0.3276],
        [0.3192],
        [0.3225],
        [0.3426],
        [0.3286]], grad_fn=<SigmoidBackward0>)

In [199]:
# access layers inside Sequential by index:
# first Linear layer's weights 
# (index 0=Linear, 1=ReLU, 2=Linear, 3=Sigmoid)
model2.network[0].weight 

Parameter containing:
tensor([[ 0.4436, -0.0258,  0.0610,  0.1570,  0.0890],
        [-0.2435,  0.0092, -0.3564,  0.0798,  0.2930],
        [-0.2909, -0.2491,  0.3820,  0.1913,  0.1368]], requires_grad=True)

In [200]:
# second Linear layer's weights 
model2.network[2].weight

Parameter containing:
tensor([[-0.1577, -0.2806,  0.0710]], requires_grad=True)

In [201]:
#see ALL parameters:
for name, param in model2.named_parameters():
    print(name, param.shape)

network.0.weight torch.Size([3, 5])
network.0.bias torch.Size([3])
network.2.weight torch.Size([1, 3])
network.2.bias torch.Size([1])


In [203]:
summary(model2, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model2                                   [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00